In [ ]:
########## This code is used to pull meteorological predictor variable data from Daymet for all profiles ##########

In [ ]:
# Import the Packages
import ee
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
import numpy as np
from collections import defaultdict
from tqdm.notebook import tqdm
ee.Initialize()

In [ ]:
############# Load in Base Data #############

In [ ]:
### Pull in SWORD Lines for GEE ###
SWORD_Nodes = ee.FeatureCollection('Insert_GEE_Asset_Path_Obstructed_Nodes_Base_Clean') # Update this file path

In [ ]:
###  Pull in the Dam File ###
Dams = gpd.read_file(r"F:\Insert_File_Path_of_Shapefile\Study_Dams.shp") # Update this file path

In [ ]:
### All Potential Pull in Profiles of Interest ###
ProfilesOfInterest = pd.read_csv(r"F:\Insert_File_Path_Here\Usable_Profiles_List.csv") # Update this file path
ProfilesOfInterest

In [ ]:
## Creat a list of the Dams
DamsList = ProfilesOfInterest["Assgn_dam"].unique().tolist()
DamsList.sort()

In [ ]:
##################################################
### Pull Daymet Data (Temps, radiation, vapor pressure ) ###
#################################################

In [ ]:
### Set Imagery Parameters ###
#Add Earth Engine dataset
DaymetProduct = ee.ImageCollection("NASA/ORNL/DAYMET_V4") 

# Set parameters for the images
date_start = ee.Date('2013-03-01')
date_end = ee.Date('2024-12-31')

## Filter Imagery Product by Date
Daymet_Data = DaymetProduct.filterDate(date_start, date_end)
Daymet_Data.size().getInfo()

In [ ]:
# Fix Padded Zeros Issue
ProfilesOfInterest['Month_Pad'] = ProfilesOfInterest['Month'].astype(str)
ProfilesOfInterest['Month_Pad'] = ProfilesOfInterest['Month_Pad'].str.pad(width=2, side='left', fillchar='0')
ProfilesOfInterest['Day_Pad'] = ProfilesOfInterest['Day'].astype(str)
ProfilesOfInterest['Day_Pad'] = ProfilesOfInterest['Day_Pad'].str.pad(width=2, side='left', fillchar='0')

## Create ID Code
cols = ['Year', 'Month_Pad', 'Day_Pad']
ProfilesOfInterest['IDString'] = ProfilesOfInterest[cols].apply(lambda row: ''.join(row.values.astype(str)), axis=1)

In [ ]:
## Filter Images to Match Profiles
ImageList = ProfilesOfInterest["IDString"].unique().tolist()
Daymet_Filter = Daymet_Data.filter(ee.Filter.inList('system:index', ImageList))
Daymet_Filter.size().getInfo()

In [ ]:
## Define the Functions
# Create Functions for extracting image info
def addDate(image):
    image_date = ee.Date(image.date())
    image_date = ee.Number.parse(image_date.format('YYYYMMdd'))
    return image.addBands(ee.Image(image_date).rename('date').toInt())

def rasterExtraction(image):
    feature = image.sampleRegions(
        collection = SWORDSubset,
        scale = 2500
    )
    return feature

In [ ]:
### The following process can be a little slow. So, it can be a good idea to split the dam list up and run it in sections. 
# This could even be done by creating several copies of the script and running them concurrently. 

In [ ]:
## Create Empty Dataframe to join to
Daymet_Data_all = pd.DataFrame()

for i in tqdm(DamsList[0:25]): ## Update the list here
    SWORDSubset = SWORD_Nodes.filter(ee.Filter.eq('Assgn_dam', i))
    ProfileSubset = ProfilesOfInterest[(ProfilesOfInterest["Assgn_dam"] == i)]
    ListofProfileDates = ProfileSubset["IDString"].unique().tolist()
    Daymet_Filter_Subset = Daymet_Filter.filter(ee.Filter.inList('system:index', ListofProfileDates))

    # Extract 
    DamDaymet = Daymet_Filter_Subset.select(['srad', 'tmax', 'vp']).map(addDate).map(rasterExtraction).flatten()

    if DamDaymet.size().getInfo() > 0: 
        # Get Column Info
        Column_df = list(SWORDSubset.limit(0).getInfo()['columns'])
        Column_df.extend([ 'srad', 'tmax', 'vp','date']) 

        # Get nested list
        nested_list = DamDaymet.reduceColumns(ee.Reducer.toList(len(Column_df)),Column_df).values().get(0)
        Daymet_Loop_Data = nested_list.getInfo()

        # Create DF of data
        DamDaymet = pd.DataFrame(Daymet_Loop_Data, columns = [Column_df ])
        
        # Add to DF
        output = pd.concat([Daymet_Data_all, DamDaymet], ignore_index=True)
        
        # Save to Big DF 
        Daymet_Data_all = output

        # Print Progress
        print("Dam ", str(i), "completed.")
    
    else: 
        print("Dam ", str(i), "has no data and is skipped.")

print("Export Daymet Data -- Completed")

In [ ]:
## Save the DF to a CSV -- Note the naming if running in sections. I ended my runs with "_P#". Multiple CSVs are combined in the next set of scripts
Daymet_Data_all.to_csv(r"F:\Insert_File_Output_Path_Here\Daymet_P1.csv") # Update this file path

## Preview
Daymet_Data_all